# Chavruta.AI — LoRA משגר (Colab / Kaggle / Lightning)

מחברת **משגר דקה** בלבד — כל לוגיקת האימון ב-`train_lora.py` (מקור אמת יחיד).

⚠️ **חובה:** `Runtime → Change runtime type → **GPU (T4)**` — לא TPU ולא CPU.

💾 **Checkpoints מקומית** ב-`/content/outputs/` (זמני — תא 5 מוריד את ה-adapter).

📦 **דאטה מ-Hugging Face Hub** (repo ציבורי — אין צורך בטוקן). **מודל: `unsloth/Qwen3-4B`** (יציב ב-fp16 על T4).

### 1. ודא שיש GPU

In [ ]:
!nvidia-smi

### 2. התקנה (פעם אחת לסשן)

In [ ]:
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U trl peft accelerate bitsandbytes datasets huggingface_hub

### 3. הבא את הקוד + הדאטה מ-Hugging Face Hub

ה-repo **ציבורי** — אין צורך ב-`login()`. (אם תחזיר אותו לפרטי, בטל את ההערה מ-`login()`.)

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download
# from huggingface_hub import login; login()  # רק אם תחזיר את ה-repo לפרטי

HF_DATASET = "Yehuda-Rubin/chavruta-torah-mixed"

os.makedirs("/content/data/processed", exist_ok=True)
os.makedirs("/content/scripts", exist_ok=True)
os.chdir("/content")

downloads = {
    "torah_mixed_train.jsonl": "data/processed/torah_mixed_train.jsonl",
    "torah_mixed_val.jsonl":   "data/processed/torah_mixed_val.jsonl",
    "train_lora.py":           "scripts/train_lora.py",
}
for remote, local in downloads.items():
    p = hf_hub_download(repo_id=HF_DATASET, filename=remote, repo_type="dataset", local_dir="/content/_hf")
    shutil.copy(p, f"/content/{local}")
    print(f"✅ {os.path.getsize(f'/content/{local}')/1e6:7.2f} MB  {local}")

### 3.5 ⚠️ Patch זמני ל-`train_lora.py` (רק אם הגרסה ב-HF עדיין ישנה)

אם כבר העלית את הסקריפט המתוקן ל-HF — דלג על התא הזה. אחרת הרץ אותו **אחרי תא 3 ולפני האימון**, וודא שהפלט מראה `padding_free = False`. **אל תריץ את תא 3 שוב אחרי זה.**

In [ ]:
p = "/content/scripts/train_lora.py"
s = open(p, encoding="utf-8").read()
if "padding_free" not in s:
    s = s.replace(
        "max_seq_length          = args.max_seq,",
        "max_length              = args.max_seq,\n        padding_free            = False,",
    )
    open(p, "w", encoding="utf-8").write(s)
import subprocess
print(subprocess.run(["grep","-nE","padding_free|max_length|trainer = SFTTrainer", p],
                     capture_output=True, text=True).stdout or "⚠️ לא נמצא — בדוק ידנית")

### 4. אימון — checkpoint מקומי ב-`/content/outputs`

אם OOM: הוסף `--max_seq 1024` או `--bs 1 --grad_accum 8`.

In [ ]:
!python scripts/train_lora.py \
  --model unsloth/Qwen3-4B \
  --epochs 1 --max_seq 2048 \
  --bs 2 --grad_accum 4 \
  --out /content/outputs/chavruta-qwen3-4b-lora

### 5. הורד את ה-adapter (אחסון מקומי זמני — הורד לפני ניתוק!)

In [ ]:
import shutil
adapter_dir = "/content/outputs/chavruta-qwen3-4b-lora"
zip_path = shutil.make_archive("/content/chavruta_adapter", "zip", adapter_dir)
print("📦", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("לא Colab או אין דפדפן — הורד ידנית מהסרגל:", zip_path)

---
### הערות
- **מודל:** `unsloth/Qwen3-4B` — יציב ב-fp16 על T4 (בניגוד ל-Qwen3.5 שנפל ל-float32).
- **repo ציבורי:** אין צורך בטוקן להורדה. להעלאת עדכונים (`upload_dataset_hf.py`) עדיין צריך טוקן Write.
- **תא 3.5 (patch):** דרוש רק עד שתעלה את `train_lora.py` המתוקן ל-HF. אחרי שתעלה — מחק/דלג עליו.
- **Kaggle/Lightning:** אותה מחברת; רק תא 5 (`files.download`) הוא Colab-ספציפי.